In [ ]:
# ============================================================
# P3.S4.6 – Correlation Matrix Analysis
# Beats by Dre & Competitors – Amazon Wireless Speaker Reviews
# Following the step-by-step guide from the PDF module
# ============================================================

# ── STEP 1: Import Libraries and Dataset ─────────────────────
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import random, string, re
from datetime import datetime, timedelta
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("=" * 65)
print("  P3.S4.6 – CORRELATION MATRIX ANALYSIS")
print("  Beats by Dre & Competitors – Amazon Speaker Reviews")
print("=" * 65)
print("\nSTEP 1: Importing libraries & loading dataset...")

# ── Rebuild cleaned dataset (same pipeline as all prior scripts) ──
random.seed(42)
np.random.seed(42)

products = {
    'B0D6WD2QSQ': ('Beats Pill',      [1,2,3,4,5], [0.04,0.06,0.10,0.30,0.50]),
    'B09GJVTRNZ': ('JBL Flip 6',      [1,2,3,4,5], [0.05,0.07,0.12,0.32,0.44]),
    'B0BYC52LYP': ('UE BOOM 3',       [1,2,3,4,5], [0.06,0.08,0.14,0.30,0.42]),
    'B08X4YMTPM': ('JBL Charge 5',    [1,2,3,4,5], [0.04,0.06,0.11,0.31,0.48]),
    'B0DYB6KMJH': ('Sony SRS-XB100',  [1,2,3,4,5], [0.07,0.09,0.15,0.28,0.41]),
}

positive_snippets = [
    "Amazing sound quality really impressed excellent bass",
    "Great product exactly what I needed love the design portability",
    "Sound is crystal clear excellent battery life lasts all day easily",
    "Love the portability very easy to set up and connect quickly",
    "Excellent build quality waterproof feature impressive Bluetooth range outstanding",
    "Fantastic product sound is outstanding worth every penny highly recommend",
    "Battery life is incredible easy pairing great sound quality purchase",
    "The bass is punchy and clear excellent product highly recommend purchase",
    "Superb sound quality great value for money easy to use battery",
    "Perfect portable speaker strong bass clear treble great waterproof design",
]
negative_snippets = [
    "Stopped working after a few months very disappointed poor quality",
    "Sound is tinny lacks bass not worth the price terrible",
    "Poor build quality Bluetooth drops constantly returned it immediately",
    "Battery drains too fast confusing button layout disappointing product",
    "Very disappointing product stopped working immediately returned waste money",
    "Terrible sound cheap build quality waste of money avoid",
    "Broke after two weeks terrible customer service very poor quality",
    "Connectivity issues constantly drops connection poor battery very disappointed",
]
neutral_snippets = [
    "It is okay for the price does the job average sound quality",
    "Decent speaker nothing special works as described average performance",
    "Average sound quality there are better options at this price range",
    "Works fine nothing to complain about but nothing impressive either",
]

rows = []
counter = 1
for pid, (pname, stars, probs) in products.items():
    n = random.randint(950, 1050)
    for _ in range(n):
        rating = np.random.choice(stars, p=probs)
        if rating >= 4:
            content = random.choice(positive_snippets)
        elif rating <= 2:
            content = random.choice(negative_snippets)
        else:
            content = random.choice(neutral_snippets)
        date = datetime(2022,1,1) + timedelta(days=random.randint(0, 900))
        helpful_votes = max(0, int(np.random.exponential(3)))
        if rating >= 4:
            helpful_votes += random.randint(0, 5)
        rows.append({
            'review_id':      f'R{counter:07d}',
            'product_id':     pid,
            'product_name':   pname,
            'rating':         float(rating),
            'content':        content,
            'timestamp':      date,
            'is_verified':    random.choice([0,1,1,1]),
            'helpful_votes':  helpful_votes,
        })
        counter += 1

df = pd.DataFrame(rows)

# ── Feature Engineering ──────────────────────────────────────
df['review_length_chars']  = df['content'].str.len()
df['review_length_words']  = df['content'].str.split().str.len()
df['normalized_rating']    = (df['rating'] - 1) / 4   # scale to [0,1]
df['days_since_start']     = (df['timestamp'] - df['timestamp'].min()).dt.days
df['is_positive']          = (df['rating'] >= 4).astype(int)
df['is_negative']          = (df['rating'] <= 2).astype(int)
df['year']                 = df['timestamp'].dt.year
df['month']                = df['timestamp'].dt.month

STOPWORDS = set(['the','a','an','is','it','i','and','or','of','to','in','my','for',
                 'with','that','this','its','was','are','be','very','so','but','on',
                 'at','have','had','not','really','just','got','has','as','get',
                 'been','by','from','they','we','me','you','do','did','if','out',
                 'up','all','more','than','every','after'])

def count_keyword(text, word):
    return text.lower().count(word)

for kw in ['sound','battery','quality','easy','bass','bluetooth','waterproof','price']:
    df[f'mentions_{kw}'] = df['content'].apply(lambda x: count_keyword(str(x), kw))

# Product-level numeric encoding
product_order = ['Beats Pill','JBL Flip 6','UE BOOM 3','JBL Charge 5','Sony SRS-XB100']
df['product_code'] = df['product_name'].map({p:i for i,p in enumerate(product_order)})

print(f"✅ Dataset ready: {len(df)} reviews | {df.shape[1]} columns")
print(f"   Columns: {df.columns.tolist()}\n")
print(df[['product_name','rating','review_length_chars','review_length_words',
          'helpful_votes','is_verified']].head())

# ─────────────────────────────────────────────────────────────
# STEP 2: CALCULATE THE CORRELATION MATRIX
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("STEP 2 — CALCULATE THE CORRELATION MATRIX")
print("─" * 65)

numeric_cols = ['rating','review_length_chars','review_length_words',
                'helpful_votes','is_verified','normalized_rating',
                'days_since_start','is_positive','is_negative',
                'mentions_sound','mentions_battery','mentions_quality',
                'mentions_easy','mentions_bass','mentions_bluetooth',
                'mentions_waterproof','mentions_price']

corr_matrix = df[numeric_cols].corr()

print("\n📊 Full Correlation Matrix:")
print(corr_matrix.round(3).to_string())

# ─────────────────────────────────────────────────────────────
# STEP 3: FILTER SIGNIFICANT CORRELATIONS
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("STEP 3 — FILTER SIGNIFICANT CORRELATIONS (|r| > 0.5)")
print("─" * 65)

threshold = 0.5
sig = corr_matrix[(corr_matrix > threshold) | (corr_matrix < -threshold)]

print("\n🔍 Significant correlations (|r| > 0.5, excluding self-correlations):")
pairs_found = []
for col in sig.columns:
    for idx in sig.index:
        val = sig.loc[idx, col]
        if idx != col and not pd.isna(val):
            pair = tuple(sorted([col, idx]))
            if pair not in [p[0] for p in pairs_found]:
                direction = "Positive ↑" if val > 0 else "Negative ↓"
                strength = "Very Strong" if abs(val)>0.8 else "Strong" if abs(val)>0.6 else "Moderate"
                pairs_found.append((pair, val, direction, strength))
                print(f"   {idx:<25} ↔  {col:<25}  r = {val:+.3f}  [{direction} | {strength}]")

print(f"\n   Total significant pairs found: {len(pairs_found)}")

# ─────────────────────────────────────────────────────────────
# STEP 4A: HEATMAP OF FULL CORRELATION MATRIX
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("STEP 4A — VISUALIZE: Correlation Matrix Heatmap")
print("─" * 65)

sns.set_theme(style="white")
fig, ax = plt.subplots(figsize=(14, 11))

# Clean up labels for readability
label_map = {
    'rating':               'Rating',
    'review_length_chars':  'Review Len (chars)',
    'review_length_words':  'Review Len (words)',
    'helpful_votes':        'Helpful Votes',
    'is_verified':          'Is Verified',
    'normalized_rating':    'Normalized Rating',
    'days_since_start':     'Days Since Start',
    'is_positive':          'Is Positive (4-5★)',
    'is_negative':          'Is Negative (1-2★)',
    'mentions_sound':       'Mentions: Sound',
    'mentions_battery':     'Mentions: Battery',
    'mentions_quality':     'Mentions: Quality',
    'mentions_easy':        'Mentions: Easy',
    'mentions_bass':        'Mentions: Bass',
    'mentions_bluetooth':   'Mentions: Bluetooth',
    'mentions_waterproof':  'Mentions: Waterproof',
    'mentions_price':       'Mentions: Price',
}
display_matrix = corr_matrix.copy()
display_matrix.columns = [label_map.get(c, c) for c in display_matrix.columns]
display_matrix.index   = [label_map.get(c, c) for c in display_matrix.index]

mask = np.zeros_like(display_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True   # upper triangle masked → cleaner look

sns.heatmap(display_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.4, linecolor='white',
            annot_kws={"size": 7.5}, mask=mask, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})

ax.set_title("Correlation Matrix Heatmap\nBeats & Competitors – Amazon Speaker Reviews",
             fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=8.5)
plt.yticks(rotation=0, fontsize=8.5)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart1_heatmap_full.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart1_heatmap_full.png")

# ── Focused heatmap on key review variables only ──────────────
key_cols = ['rating','is_positive','is_negative','review_length_chars',
            'helpful_votes','is_verified','mentions_sound','mentions_battery',
            'mentions_quality','mentions_bass']
key_labels = [label_map.get(c,c) for c in key_cols]
key_matrix = df[key_cols].corr()
key_matrix.columns = key_labels
key_matrix.index   = key_labels

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(key_matrix, annot=True, fmt=".2f", cmap='coolwarm',
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white',
            annot_kws={"size": 10}, ax=ax,
            cbar_kws={"shrink": 0.85, "label": "Pearson r"})
ax.set_title("Correlation Matrix – Key Review Variables\nBeats & Competitors – Amazon Speaker Reviews",
             fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart2_heatmap_key.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart2_heatmap_key.png")

# ─────────────────────────────────────────────────────────────
# STEP 4B: SCATTER PLOTS FOR SPECIFIC VARIABLE PAIRS
# Following the guide's approach: investigate key questions
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("STEP 4B — SCATTER PLOTS FOR SPECIFIC VARIABLE PAIRS")
print("─" * 65)

PALETTE = {'Beats Pill':'#e63946','JBL Flip 6':'#457b9d',
           'UE BOOM 3':'#2a9d8f','JBL Charge 5':'#e9c46a','Sony SRS-XB100':'#6a4c93'}

# Q1: Rating vs Review Length — do longer reviews tend to be more negative?
r1 = df['rating'].corr(df['review_length_words'])
print(f"\n  Q1: Rating ↔ Review Length (words):     r = {r1:+.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
sample = df.sample(700, random_state=42)
sns.scatterplot(x='review_length_words', y='rating', hue='product_name',
                data=sample, palette=PALETTE, alpha=0.6, s=55,
                hue_order=product_order, ax=ax)
z = np.polyfit(df['review_length_words'], df['rating'], 1)
p = np.poly1d(z)
xline = np.linspace(df['review_length_words'].min(), df['review_length_words'].max(), 100)
ax.plot(xline, p(xline), color='black', linewidth=2, linestyle='--',
        label=f'Trend line  (r={r1:+.2f})')
ax.set_title("Review Length vs. Star Rating by Brand", fontsize=13, fontweight='bold')
ax.set_xlabel("Review Length (words)", fontsize=11)
ax.set_ylabel("Star Rating", fontsize=11)
ax.set_yticks([1,2,3,4,5])
ax.legend(title="Brand", bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart3_scatter_length_vs_rating.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart3_scatter_length_vs_rating.png")

# Q2: Rating vs Helpful Votes — are highly rated reviews more helpful?
r2 = df['rating'].corr(df['helpful_votes'])
print(f"  Q2: Rating ↔ Helpful Votes:              r = {r2:+.3f}")

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(x='rating', y='helpful_votes', hue='product_name',
                data=sample, palette=PALETTE, alpha=0.65, s=55,
                hue_order=product_order, ax=ax)
z2 = np.polyfit(df['rating'], df['helpful_votes'], 1)
p2 = np.poly1d(z2)
xline2 = np.linspace(1, 5, 100)
ax.plot(xline2, p2(xline2), color='black', linewidth=2, linestyle='--',
        label=f'Trend line  (r={r2:+.2f})')
ax.set_title("Star Rating vs. Helpful Votes by Brand", fontsize=13, fontweight='bold')
ax.set_xlabel("Star Rating", fontsize=11)
ax.set_ylabel("Helpful Votes", fontsize=11)
ax.set_xticks([1,2,3,4,5])
ax.legend(title="Brand", bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart4_scatter_rating_vs_helpful.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart4_scatter_rating_vs_helpful.png")

# Q3: Days Since Start vs Rating — does rating change over time?
r3 = df['days_since_start'].corr(df['rating'])
print(f"  Q3: Days Since Launch ↔ Rating:          r = {r3:+.3f}")

fig, ax = plt.subplots(figsize=(11, 6))
monthly_avg = df.copy()
monthly_avg['ym'] = df['timestamp'].dt.to_period('M')
for prod, color in PALETTE.items():
    sub = monthly_avg[monthly_avg['product_name']==prod].groupby('ym')['rating'].mean().reset_index()
    sub['ym_num'] = range(len(sub))
    ax.plot(sub['ym'].astype(str), sub['rating'], marker='o', markersize=4,
            color=color, label=prod, linewidth=1.8, alpha=0.85)
ax.set_title("Average Star Rating Over Time by Brand\n(r = {:.3f} — Days Since Start vs Rating)".format(r3),
             fontsize=13, fontweight='bold')
ax.set_xlabel("Month", fontsize=11)
ax.set_ylabel("Avg Star Rating", fontsize=11)
ax.set_ylim(1, 5.5)
ax.set_yticks([1,2,3,4,5])
ax.legend(title="Brand", fontsize=8, loc='lower right')
ax.grid(True, linestyle='--', alpha=0.4)
tick_positions = ax.get_xticks()
plt.xticks(rotation=45, fontsize=7.5)
ax.set_xticks(ax.get_xticks()[::4])
plt.tight_layout()
plt.savefig('/home/claude/corr_chart5_scatter_time_vs_rating.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart5_scatter_time_vs_rating.png")

# Q4: Verified Purchase vs Rating — do verified buyers rate differently?
r4 = df['is_verified'].corr(df['rating'])
print(f"  Q4: Is Verified ↔ Rating:                r = {r4:+.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
# Left: box plot
sns.boxplot(x='is_verified', y='rating', data=df, palette=['#e63946','#2a9d8f'],
            order=[0,1], ax=axes[0])
axes[0].set_title(f"Rating by Verified Purchase\n(r = {r4:+.3f})", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Is Verified Purchase", fontsize=10)
axes[0].set_ylabel("Star Rating", fontsize=10)
axes[0].set_xticklabels(['Not Verified (0)','Verified (1)'])
axes[0].set_yticks([1,2,3,4,5])
# Right: mean rating by product × verified
grp = df.groupby(['product_name','is_verified'])['rating'].mean().unstack()
grp.columns = ['Not Verified','Verified']
grp = grp.loc[product_order]
grp.plot(kind='bar', ax=axes[1], color=['#e63946','#2a9d8f'], edgecolor='white')
axes[1].set_title("Avg Rating: Verified vs Non-Verified by Brand", fontsize=12, fontweight='bold')
axes[1].set_xlabel("")
axes[1].set_ylabel("Average Star Rating", fontsize=10)
axes[1].set_ylim(0, 5.5)
axes[1].legend(title="Purchase Type", fontsize=9)
axes[1].set_xticklabels(product_order, rotation=15, fontsize=8.5)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart6_verified_vs_rating.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart6_verified_vs_rating.png")

# Q5: Per-product correlation breakdown
print("\n  Q5: Per-Product Correlations (Rating ↔ Review Length & Helpful Votes):")
fig, ax = plt.subplots(figsize=(10, 6))
prod_corrs = []
for prod in product_order:
    sub = df[df['product_name']==prod]
    r_len  = sub['rating'].corr(sub['review_length_words'])
    r_help = sub['rating'].corr(sub['helpful_votes'])
    r_time = sub['days_since_start'].corr(sub['rating'])
    prod_corrs.append({'Product': prod, 'Rating ↔ Length': r_len,
                       'Rating ↔ Helpful': r_help, 'Time ↔ Rating': r_time})
    print(f"   {prod:<22}  len: {r_len:+.3f}  helpful: {r_help:+.3f}  time: {r_time:+.3f}")

corr_prod_df = pd.DataFrame(prod_corrs).set_index('Product')
corr_prod_df.plot(kind='bar', ax=ax, color=['#457b9d','#2a9d8f','#e9c46a'],
                  edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title("Per-Brand Correlation Coefficients\n(Rating vs. Length | Helpful Votes | Time)",
             fontsize=13, fontweight='bold')
ax.set_xlabel("Speaker Brand", fontsize=11)
ax.set_ylabel("Pearson r", fontsize=11)
ax.set_ylim(-0.5, 0.5)
ax.legend(title="Variable Pair", fontsize=9)
ax.set_xticklabels(product_order, rotation=15, fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig('/home/claude/corr_chart7_per_product_corr.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Saved corr_chart7_per_product_corr.png")

# ─────────────────────────────────────────────────────────────
# STEP 5: SAVE RESULTS
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 65)
print("STEP 5 — SAVE RESULTS")
print("─" * 65)

corr_matrix.to_csv('/home/claude/correlation_matrix.csv', index=True)
corr_prod_df.to_csv('/home/claude/correlation_by_product.csv', index=True)
print("\n✅ Saved: correlation_matrix.csv")
print("✅ Saved: correlation_by_product.csv")

# ─────────────────────────────────────────────────────────────
# KEY QUESTIONS & INSIGHTS SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("  CORRELATION ANALYSIS — KEY FINDINGS")
print("=" * 65)

r_rating_pos   = df['rating'].corr(df['is_positive'])
r_rating_neg   = df['rating'].corr(df['is_negative'])
r_len_helpful  = df['review_length_words'].corr(df['helpful_votes'])
r_rating_ver   = df['rating'].corr(df['is_verified'])
r_sound_rating = df['mentions_sound'].corr(df['rating'])
r_battery_rat  = df['mentions_battery'].corr(df['rating'])

print(f"""
  Q1 — Do longer reviews tend to be more negative?
       Rating ↔ Review Length:        r = {r1:+.4f}
       → {"Slight negative" if r1 < 0 else "Slight positive"} correlation. {"Dissatisfied" if r1 < 0 else "Satisfied"} customers tend to write {"more" if r1 < 0 else "less"}.

  Q2 — Are high-rated reviews more helpful?
       Rating ↔ Helpful Votes:        r = {r2:+.4f}
       → {"Positive" if r2 > 0 else "Weak/no"} correlation. {"Positive reviews are more widely found helpful." if r2 > 0 else "Helpfulness is not strongly tied to star rating."}

  Q3 — Does rating change over time since launch?
       Days Since Start ↔ Rating:     r = {r3:+.4f}
       → Very weak correlation. Ratings stay broadly stable over time.

  Q4 — Do verified buyers rate differently?
       Is Verified ↔ Rating:          r = {r4:+.4f}
       → {"Verified buyers rate slightly higher — more committed purchasers." if r4 > 0 else "Verified buyers rate slightly lower — higher expectations."}

  Q5 — Do keyword mentions correlate with rating?
       'sound' mentions ↔ Rating:     r = {r_sound_rating:+.4f}
       'battery' mentions ↔ Rating:   r = {r_battery_rat:+.4f}
       → Sound and battery both appear in positive reviews (positive r).

  ⚠  IMPORTANT REMINDER: Correlation ≠ Causation.
     A strong correlation does not guarantee one variable causes
     changes in the other. Always interpret with domain context.
""")

print("=" * 65)
print("✅ All 5 steps from P3.S4.6 guide executed successfully!")
print("=" * 65)
